# Phase 3: Representation Gap Analysis

Goals:
- distances in the Graphormer embedding space
- MMD distance (distribution-alignment view)
- Wasserstein distance
- CORAL distance
- representation similarity

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import pickle
import json
from scipy.stats import wasserstein_distance, ks_2samp
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
FEAT_DIR = os.path.join(ROOT, 'features')
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 3: Representation Gap Analysis")
print("="*100)
print(f"Results: {RESULTS_DIR}\n")
np.random.seed(42)  # reproducible sampling

PHASE 3: Representation Gap Analysis
Results: ..\benchmark\results



In [2]:
# Load Phase 1 & 2 results
print("Loading Phase 1 & 2 results...\n")

with open(os.path.join(RESULTS_DIR, 'preprocessed_data.pkl'), 'rb') as f:
    phase1_data = pickle.load(f)

with open(os.path.join(RESULTS_DIR, 'chemical_space_data.pkl'), 'rb') as f:
    chemical_data = pickle.load(f)

source_data = phase1_data['source_data']
target_data = phase1_data['target_data']
PROPERTIES = phase1_data['properties']
FOLD = phase1_data['fold']

print(f"Properties: {PROPERTIES}")
print(f"Fold: {FOLD}\n")

Loading Phase 1 & 2 results...

Properties: ['clearance', 'half_life', 'fu']
Fold: 0



In [3]:
# Load Graphormer embeddings
print("Loading Graphormer embeddings...\n")

emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']

smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"✓ Loaded {len(smiles_to_emb)} embeddings")
print(f"  Embedding dimension: {emb_X.shape[1]}\n")

Loading Graphormer embeddings...

✓ Loaded 58304 embeddings
  Embedding dimension: 768



In [4]:
def gaussian_kernel(X, Y=None, sigma=1.0):
    """Compute Gaussian kernel"""
    if Y is None:
        Y = X
    
    # Euclidean distance squared
    X_sqnorm = np.sum(X**2, axis=1, keepdims=True)
    Y_sqnorm = np.sum(Y**2, axis=1, keepdims=True).T
    XY = np.dot(X, Y.T)
    dist_sq = np.maximum(X_sqnorm + Y_sqnorm - 2*XY, 0)
    
    return np.exp(-dist_sq / (2 * sigma**2))

def compute_mmd(X_src, X_tgt, sigma=1.0):
    """Maximum Mean Discrepancy"""
    K_ss = gaussian_kernel(X_src, X_src, sigma)
    K_tt = gaussian_kernel(X_tgt, X_tgt, sigma)
    K_st = gaussian_kernel(X_src, X_tgt, sigma)
    
    n_s = X_src.shape[0]
    n_t = X_tgt.shape[0]
    
    mmd_loss = (1.0 / (n_s**2)) * np.sum(K_ss) + \
               (1.0 / (n_t**2)) * np.sum(K_tt) - \
               (2.0 / (n_s * n_t)) * np.sum(K_st)
    
    return np.sqrt(np.maximum(mmd_loss, 0))

def compute_coral(X_src, X_tgt):
    """CORAL distance: covariance alignment"""
    # Center data
    X_src_centered = X_src - np.mean(X_src, axis=0, keepdims=True)
    X_tgt_centered = X_tgt - np.mean(X_tgt, axis=0, keepdims=True)
    
    # Covariance matrices
    cov_src = np.dot(X_src_centered.T, X_src_centered) / X_src.shape[0]
    cov_tgt = np.dot(X_tgt_centered.T, X_tgt_centered) / X_tgt.shape[0]
    
    # Frobenius norm
    coral_loss = np.sum((cov_src - cov_tgt)**2)
    
    return np.sqrt(coral_loss)

def compute_wasserstein_sliced(X_src, X_tgt, n_directions=100):
    """Sliced Wasserstein distance (approximation)"""
    distances = []
    
    for _ in range(n_directions):
        # Random direction
        theta = np.random.randn(X_src.shape[1])
        theta = theta / np.linalg.norm(theta)
        
        # Project data
        proj_src = np.dot(X_src, theta)
        proj_tgt = np.dot(X_tgt, theta)
        
        # 1D Wasserstein
        proj_src_sorted = np.sort(proj_src)
        proj_tgt_sorted = np.sort(proj_tgt)
        
        # Interpolate to same size
        n = max(len(proj_src_sorted), len(proj_tgt_sorted))
        if len(proj_src_sorted) < n:
            proj_src_sorted = np.interp(np.linspace(0, len(proj_src_sorted)-1, n), 
                                         np.arange(len(proj_src_sorted)), proj_src_sorted)
        if len(proj_tgt_sorted) < n:
            proj_tgt_sorted = np.interp(np.linspace(0, len(proj_tgt_sorted)-1, n), 
                                         np.arange(len(proj_tgt_sorted)), proj_tgt_sorted)
        
        wd = np.mean(np.abs(proj_src_sorted - proj_tgt_sorted))
        distances.append(wd)
    
    return np.mean(distances)

print("✓ Distance functions defined")

✓ Distance functions defined


In [5]:
# Extract embeddings for each property
representation_results = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    representation_results[prop] = {}
    
    # Get embeddings for source and target
    src_data = source_data[prop]
    tgt_data = target_data[prop]
    
    # Extract embeddings
    X_src_list = []
    valid_src_smiles = []
    for idx, row in src_data.iterrows():
        smiles = row['smiles_original']  # embedding keys are the original SMILES from the CV csv
        if smiles in smiles_to_emb:
            X_src_list.append(smiles_to_emb[smiles])
            valid_src_smiles.append(smiles)
    
    X_tgt_list = []
    valid_tgt_smiles = []
    for idx, row in tgt_data.iterrows():
        smiles = row['smiles_original']  # embedding keys are the original SMILES from the CV csv
        if smiles in smiles_to_emb:
            X_tgt_list.append(smiles_to_emb[smiles])
            valid_tgt_smiles.append(smiles)
    
    if len(X_src_list) > 0 and len(X_tgt_list) > 0:
        X_src = np.array(X_src_list, dtype='float32')
        X_tgt = np.array(X_tgt_list, dtype='float32')
        
        print(f"  Source embeddings: {X_src.shape}")
        print(f"  Target embeddings: {X_tgt.shape}")
        
        # Sample if too large (for speed)
        max_samples = 2000
        if len(X_src) > max_samples:
            idx = np.random.choice(len(X_src), max_samples, replace=False)
            X_src = X_src[idx]
            valid_src_smiles = [valid_src_smiles[i] for i in idx]
        
        if len(X_tgt) > max_samples:
            idx = np.random.choice(len(X_tgt), max_samples, replace=False)
            X_tgt = X_tgt[idx]
            valid_tgt_smiles = [valid_tgt_smiles[i] for i in idx]
        
        print(f"  (After sampling if needed)")
        print(f"    Source: {X_src.shape}")
        print(f"    Target: {X_tgt.shape}")
        
        # Compute distances
        print(f"  Computing distances...")
        
        mmd_dist = compute_mmd(X_src, X_tgt, sigma=1.0)
        # With σ=1 the kernel is nearly 0 on 768-d embeddings → compute once more with the median-heuristic σ
        Z = np.vstack([X_src[:1000], X_tgt[:1000]]).astype('float64')
        sq = np.sum(Z**2, axis=1)
        d2 = np.maximum(sq[:, None] + sq[None, :] - 2 * Z @ Z.T, 0)
        sigma_med = float(np.sqrt(np.median(d2[np.triu_indices(len(Z), 1)]) / 2))
        mmd_med = compute_mmd(X_src, X_tgt, sigma=sigma_med)
        coral_dist = compute_coral(X_src, X_tgt)
        
        print(f"    MMD distance: {mmd_dist:.4f}")
        print(f"    CORAL distance: {coral_dist:.4f}")
        
        # Sliced Wasserstein
        print(f"  Computing sliced Wasserstein (100 directions)...")
        sw_dist = compute_wasserstein_sliced(X_src, X_tgt, n_directions=100)
        print(f"    Sliced Wasserstein distance: {sw_dist:.4f}")
        
        # Mean and std of embeddings
        print(f"  Embedding statistics:")
        print(f"    Source: mean={np.mean(X_src):.4f}, std={np.std(X_src):.4f}")
        print(f"    Target: mean={np.mean(X_tgt):.4f}, std={np.std(X_tgt):.4f}")
        
        representation_results[prop] = {
            'n_source': int(len(X_src)),
            'n_target': int(len(X_tgt)),
            'mmd_distance': float(mmd_dist),
            'mmd_median_heuristic': float(mmd_med),
            'sigma_median': sigma_med,
            'coral_distance': float(coral_dist),
            'sliced_wasserstein': float(sw_dist),
            'source_mean': float(np.mean(X_src)),
            'source_std': float(np.std(X_src)),
            'target_mean': float(np.mean(X_tgt)),
            'target_std': float(np.std(X_tgt)),
        }
    else:
        print(f"  No valid embeddings")
        representation_results[prop] = {}


Property: CLEARANCE
  Source embeddings: (14386, 768)
  Target embeddings: (1027, 768)
  (After sampling if needed)
    Source: (2000, 768)
    Target: (1027, 768)
  Computing distances...
    MMD distance: 0.0390
    CORAL distance: 64.0824
  Computing sliced Wasserstein (100 directions)...
    Sliced Wasserstein distance: 0.2392
  Embedding statistics:
    Source: mean=-0.0090, std=1.0148
    Target: mean=-0.0049, std=1.0087

Property: HALF_LIFE
  Source embeddings: (9202, 768)
  Target embeddings: (1164, 768)
  (After sampling if needed)
    Source: (2000, 768)
    Target: (1164, 768)
  Computing distances...
    MMD distance: 0.0392
    CORAL distance: 57.0978
  Computing sliced Wasserstein (100 directions)...
    Sliced Wasserstein distance: 0.2193
  Embedding statistics:
    Source: mean=-0.0088, std=1.0107
    Target: mean=-0.0051, std=1.0097

Property: FU
  Source embeddings: (3778, 768)
  Target embeddings: (353, 768)
  (After sampling if needed)
    Source: (2000, 768)
    T

In [6]:
# Summary Report
print(f"\n{'='*100}")
print("REPRESENTATION GAP SUMMARY")
print(f"{'='*100}\n")

summary_data = []

for prop in PROPERTIES:
    res = representation_results[prop]
    row = {
        'Property': prop,
        'MMD': f"{res.get('mmd_distance', 0):.4f}",
        'CORAL': f"{res.get('coral_distance', 0):.4f}",
        'SlicedWasserstein': f"{res.get('sliced_wasserstein', 0):.4f}",
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()

# Interpretation: small distance = small representation gap = potentially good transferability
print("\nInterpretation:")
print("  Smaller distance = smaller representation gap = better transferability potential")
print("  (Lower values mean source and target representations are more similar)\n")


REPRESENTATION GAP SUMMARY

 Property    MMD   CORAL SlicedWasserstein
clearance 0.0390 64.0824            0.2392
half_life 0.0392 57.0978            0.2193
       fu 0.0583 57.3944            0.2086


Interpretation:
  Smaller distance = smaller representation gap = better transferability potential
  (Lower values mean source and target representations are more similar)



In [7]:
# Save results
with open(os.path.join(RESULTS_DIR, 'phase_03_representation_gap.json'), 'w') as f:
    json.dump(representation_results, f, indent=2)

print("✓ Results saved to benchmark/results/")
print(f"  - phase_03_representation_gap.json")
print("\nPhase 3 Complete! Ready for Phase 4: Label Correspondence Analysis")

✓ Results saved to benchmark/results/
  - phase_03_representation_gap.json

Phase 3 Complete! Ready for Phase 4: Label Correspondence Analysis
